In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))  # This brings 'src' into the path

In [ ]:

import yaml
config = yaml.safe_load(open('../config.yaml', 'r'))

import os
os.environ["CUDA_VISIBLE_DEVICES"] = '6'
import torch
from src.model.models_dsfno_3d import DSFNO
from src.dataloader.dataloader_3d import dataset_sr
import numpy as np
from matplotlib.colors import LogNorm
import matplotlib.pyplot as plt
import random
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
dsfno_model = DSFNO(in_channel=5, 
                    modes=config['dsfno']['modes'],
                    n_channels=config['dsfno']['n_channels'],
                    n_residual_blocks=config['dsfno']['n_residual_blocks'],
                    n_operator_blocks=config['dsfno']['n_operator_blocks'], 
                    apply_constraint=config['dsfno']['apply_constraint']).to(device)

In [ ]:
max_samples = 30
dataset = dataset_sr(max_samples=max_samples)

In [ ]:
hr_state, lr_state_tensor = dataset[2]

lr_state_tensor = lr_state_tensor.to(device)

print(f"Memory before model: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


In [ ]:
def profile_3d_forward_pass(mode=32):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
    print(f"\n=== 3D DSFNO Forward Pass Profile (modes={mode}) ===")
    
    # Create model
    dsfno = DSFNO(in_channel=5, modes=mode, n_channels=16, 
                  n_residual_blocks=2, n_operator_blocks=4, 
                  apply_constraint=True).to(device)
    
    print(f"After model creation: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    
    # Prepare input
    x = torch.unsqueeze(lr_state_tensor, 0)  # [1, 5, 32, 32, 32]
    upsample_factor = 4
    
    with torch.no_grad():
        print(f"Input shape: {x.shape}")
        print(f"After input preparation: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
        
        # Conv1: 5 -> 16 channels
        out = dsfno.conv1(x)
        print(f"After conv1: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
        print(f"Conv1 output shape: {out.shape}")
        
        # Residual blocks
        for i, layer in enumerate(dsfno.res_blocks):
            out = layer(out)
            print(f"After res_block_{i}: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
        
        # Conv2
        out = dsfno.conv2(out)
        print(f"After conv2: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
        print(f"Before 3D upsampling shape: {out.shape}")
        
        # THE MEMORY KILLER: 3D upsampling from [1,16,32,32,32] to [1,16,128,128,128]
        out = torch.nn.functional.interpolate(out, scale_factor=upsample_factor, mode='trilinear', align_corners=False)
        print(f"After 3D upsampling: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
        print(f"After 3D upsampling shape: {out.shape}")
        
        # FNO blocks (where modes parameter matters)
        for i, layer in enumerate(dsfno.fno_blocks):
            before_fno = torch.cuda.memory_allocated() / 1024**2
            out = layer(out)
            after_fno = torch.cuda.memory_allocated() / 1024**2
            print(f"FNO block {i}: {before_fno:.2f} -> {after_fno:.2f} MB (delta: {after_fno-before_fno:.2f})")
        
        # Permute for linear layers: [1,16,128,128,128] -> [1,128,128,128,16]
        out = out.permute(0, 2, 3, 4, 1)
        print(f"After permute for linear: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
        print(f"Shape before linear: {out.shape}")
        
        # FC1: [1,128,128,128,16] -> [1,128,128,128,128] - ANOTHER MEMORY KILLER
        out = dsfno.fc1(out)
        print(f"After fc1: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
        print(f"FC1 output shape: {out.shape}")
        
        out = F.gelu(out)
        print(f"After GELU: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
        
        # FC2: [1,128,128,128,128] -> [1,128,128,128,5]
        out = dsfno.fc2(out)
        print(f"After fc2: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
        print(f"FC2 output shape: {out.shape}")
        
        # Permute back: [1,128,128,128,5] -> [1,5,128,128,128]
        out = out.permute(0, 4, 1, 2, 3)
        print(f"After final permute: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
        
        # Constraint (if applied)
        if dsfno.apply_constraint:
            out = dsfno.constraint(x, out, upsample_factor)
            print(f"After constraint: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    
    print(f"Peak memory: {torch.cuda.max_memory_allocated() / 1024**2:.2f} MB")
    
    del dsfno
    return out

# Test it
output = profile_3d_forward_pass(32)
del output

In [ ]:
# Memory profiling through each step
hr_state, lr_state_tensor = dataset[2]
lr_state_tensor = lr_state_tensor.to(device)

def profile_forward_pass(mode=32):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
    print(f"\n=== Detailed Forward Pass Profile (modes={mode}) ===")
    
    # Create model
    dsfno = DSFNO(in_channel=5, modes=16, n_channels=32, 
                  n_residual_blocks=8, n_operator_blocks=8, 
                  apply_constraint=True).to(device)
    
    print(f"After model creation: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    
    # Prepare input
    x = torch.unsqueeze(lr_state_tensor, 0)
    upsample_factor = 4
    
    # Let's manually step through the forward pass
    # Initial processing
    
    # Conv1
    out = dsfno.conv1(x)
    print(f"After conv1: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    print(f"Conv1 output shape: {out.shape}")
    
    # Residual blocks
    for i, layer in enumerate(dsfno.res_blocks):
        out = layer(out)
        print(f"After res_block_{i}: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    
    # Conv2
    out = dsfno.conv2(out)
    print(f"After conv2: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    print(f"Before upsampling shape: {out.shape}")
    
    # THE BIG ONE: Upsampling
    out = torch.nn.functional.interpolate(out, scale_factor=upsample_factor, mode='trilinear', align_corners=False)
    print(f"After upsampling: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    print(f"After upsampling shape: {out.shape}")
    
    # FNO blocks (this is where modes matters)
    for i, layer in enumerate(dsfno.fno_blocks):
        out = layer(out)
        print(f"After fno_block_{i}: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    
    # Permute for linear layers
    out = out.permute(0, 2, 3, 4, 1)
    print(f"After permute for linear: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    print(f"Shape before linear: {out.shape}")
    
    # FC1 - This might be the culprit
    out = dsfno.fc1(out)
    print(f"After fc1: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    print(f"FC1 output shape: {out.shape}")
    
    out = F.gelu(out)
    print(f"After GELU: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    
    # FC2
    out = dsfno.fc2(out)
    print(f"After fc2: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    print(f"FC2 output shape: {out.shape}")
    
    # Constraint (if applied)
    if dsfno.apply_constraint:
        out = out.permute(0, 4, 1, 2, 3)
        print(f"After permute for constraint: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
        out = dsfno.constraint(x, out, upsample_factor)
        print(f"After constraint: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

    print(f"Peak memory during forward pass: {torch.cuda.max_memory_allocated() / 1024**2:.2f} MB")

    del dsfno
    return out


hr_state, lr_state_tensor = dataset[2]

lr_state_tensor = lr_state_tensor.to(device)

n_modes = [4, 8, 12, 16, 32, 64, 128]

for modes in n_modes:
    output = profile_forward_pass(modes)
    del output


In [ ]:

def profile_forward_pass(mode=32):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
    print(f"\n=== Detailed Forward Pass Profile (modes={mode}) ===")
    
    # Create model
    dsfno = DSFNO(in_channel=5, modes=16, n_channels=32, 
                  n_residual_blocks=8, n_operator_blocks=8, 
                  apply_constraint=True).to(device)
    
    print(f"After model creation: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    
    # Prepare input
    x = torch.unsqueeze(lr_state_tensor, 0)
    upsample_factor = 4
    
    # Let's manually step through the forward pass
    # Initial processing
    
    # Conv1
    out = dsfno.conv1(x)
    print(f"After conv1: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    print(f"Conv1 output shape: {out.shape}")
    
    # Residual blocks
    for i, layer in enumerate(dsfno.res_blocks):
        out = layer(out)
        print(f"After res_block_{i}: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    
    # Conv2
    out = dsfno.conv2(out)
    print(f"After conv2: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    print(f"Before upsampling shape: {out.shape}")
    
    # THE BIG ONE: Upsampling
    out = torch.nn.functional.interpolate(out, scale_factor=upsample_factor, mode='trilinear', align_corners=False)
    print(f"After upsampling: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    print(f"After upsampling shape: {out.shape}")
    
    # FNO blocks (this is where modes matters)
    for i, layer in enumerate(dsfno.fno_blocks):
        out = layer(out)
        print(f"After fno_block_{i}: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    
    # Permute for linear layers
    out = out.permute(0, 2, 3, 4, 1)
    print(f"After permute for linear: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    print(f"Shape before linear: {out.shape}")
    
    # FC1 - This might be the culprit
    out = dsfno.fc1(out)
    print(f"After fc1: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    print(f"FC1 output shape: {out.shape}")
    
    out = F.gelu(out)
    print(f"After GELU: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    
    # FC2
    out = dsfno.fc2(out)
    print(f"After fc2: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
    print(f"FC2 output shape: {out.shape}")
    
    # Constraint (if applied)
    if dsfno.apply_constraint:
        out = out.permute(0, 4, 1, 2, 3)
        print(f"After permute for constraint: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
        out = dsfno.constraint(x, out, upsample_factor)
        print(f"After constraint: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

    print(f"Peak memory during forward pass: {torch.cuda.max_memory_allocated() / 1024**2:.2f} MB")

    del dsfno
    return out


hr_state, lr_state_tensor = dataset[2]

lr_state_tensor = lr_state_tensor.to(device)

n_modes = [4, 8, 12, 16, 32, 64, 128]

for modes in n_modes:
    output = profile_forward_pass(modes)
    del output


In [ ]:

hr_state, lr_state_tensor = dataset[2]

lr_state_tensor = lr_state_tensor.to(device)

n_residual_blockss = [2, 3, 4, 5, 6]

for n_res in n_residual_blockss:
    dsfno = DSFNO(in_channel = 5,
                            modes = 16,
                            n_channels=16, 
                            n_residual_blocks=n_res, 
                            n_operator_blocks=1, 
                            apply_constraint=True).to(device)
    output = dsfno(torch.unsqueeze(lr_state_tensor,0), 4)
    memory_used = torch.cuda.memory_allocated() / 1024**2

    print(f"Memory after model: {memory_used:.2f} MB for residual blocks {n_res}")


del dsfno
del output

torch.cuda.empty_cache()



In [ ]:
hr_state, lr_state_tensor = dataset[2]

lr_state_tensor = lr_state_tensor.to(device)

op_blocks = [2, 3, 4, 5, 6]

for op_block in op_blocks:
    dsfno = DSFNO(in_channel = 5,
                            modes = 64,
                            n_channels=32, 
                            n_residual_blocks=n_res, 
                            n_operator_blocks=op_block, 
                            apply_constraint=True).to(device)
    output = dsfno(torch.unsqueeze(lr_state_tensor,0), 4)
    memory_used = torch.cuda.memory_allocated() / 1024**2

    print(f"Memory after model: {memory_used:.2f} MB for op blocks {op_block}")

    del dsfno
    del output

    torch.cuda.empty_cache()



In [ ]:

memory_used = torch.cuda.memory_allocated() / 1024**2

print(f"Memory after model: {memory_used:.2f} MB")

In [ ]:
output = dsfno(torch.unsqueeze(lr_state_tensor,0).to(device), 4)

In [ ]:
print(output.element_size() * output.nelement() / (1024 ** 2)  )
memory_used = torch.cuda.memory_allocated() / 1024**2
print(f"Memory after model: {memory_used:.2f} MB")

In [ ]:
n_plots = 3
random_list = random.sample(range(max_samples), n_plots)
scale_factor = 4
sr_factor = 2
fig, axes = plt.subplots(n_plots, 9, figsize=(20, n_plots * 3))

# plot cuts at this z level
z_level = 60
z_level_reduced = z_level // scale_factor
z_level_sr = z_level_reduced * sr_factor
for ax in axes.flat:
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in ax.spines.values():
        spine.set_visible(False)
        
for i, list_i in enumerate(random_list):
    hr_state, lr_state_tensor = dataset[list_i]
    with torch.no_grad():
        sr_state = torch.squeeze(dsfno(torch.unsqueeze(lr_state_tensor, 0).to(device), sr_factor),0).cpu().detach().numpy()
    hr_state, lr_state = hr_state.numpy(), lr_state_tensor.numpy()

    axes[i,0].imshow(lr_state[0, :, :, z_level_reduced].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())
    axes[i,0].set_ylabel(f"{list_i}")
    axes[i,1].imshow(np.sqrt(lr_state[1, :, :, z_level_reduced]**2 + lr_state[2, :, :, z_level_reduced]**2).T, origin = "lower", extent = [0, 1, 0, 1])
    axes[i,2].imshow(lr_state[4, :, :, z_level_reduced].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())
    
    axes[i,3].imshow(hr_state[0, :, :, z_level].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())
    axes[i,4].imshow(np.sqrt(hr_state[1, :, :, z_level]**2 + hr_state[2, :, :, z_level]**2).T, origin = "lower", extent = [0, 1, 0, 1])
    axes[i,5].imshow(hr_state[4, :, :, z_level].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())

    axes[i,6].imshow(sr_state[0, :, :, z_level_sr].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())
    axes[i,7].imshow(np.sqrt(sr_state[1, :, :, z_level_sr]**2 + sr_state[2, :, :, z_level_sr]**2).T, origin = "lower", extent = [0, 1, 0, 1])
    axes[i,8].imshow(sr_state[4, :, :, z_level_sr].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())

    if i == 0:
        axes[i,0].set_title("Density")
        axes[i,1].set_title("Velocity")
        axes[i,2].set_title("Pressure")
        axes[i,3].set_title("Density")
        axes[i,4].set_title("Velocity")
        axes[i,5].set_title("Pressure")
        axes[i,6].set_title("Density")
        axes[i,7].set_title("Velocity")
        axes[i,8].set_title("Pressure")

    # equal aspect ratio
    axes[i,0].set_aspect('equal', 'box')
    axes[i,1].set_aspect('equal', 'box')
    axes[i,2].set_aspect('equal', 'box')

In [ ]:
index = random.sample(range(max_samples), 1)
scale_factor = 4
sr_factor = 2
fig, axes = plt.subplots(3, 3, figsize=(12,12))

# plot cuts at this z level
z_level = 60
z_level_reduced = z_level // scale_factor
z_level_sr = z_level_reduced * sr_factor
for ax in axes.flat:
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    for spine in ax.spines.values():
        spine.set_visible(False)
        
hr_state, lr_state_tensor = dataset[list_i]

with torch.no_grad():
    sr_state = torch.squeeze(dsfno(torch.unsqueeze(lr_state_tensor, 0).to(device), sr_factor),0).cpu().detach().numpy()
hr_state, lr_state = hr_state.numpy(), lr_state_tensor.numpy()

axes[0,0].imshow(lr_state[0, :, :, z_level_reduced].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())
axes[0,0].set_ylabel(f"{list_i}")
axes[0,1].imshow(np.sqrt(lr_state[1, :, :, z_level_reduced]**2 + lr_state[2, :, :, z_level_reduced]**2).T, origin = "lower", extent = [0, 1, 0, 1])
axes[0,2].imshow(lr_state[4, :, :, z_level_reduced].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())

axes[1,0].imshow(hr_state[0, :, :, z_level].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())
axes[1,1].imshow(np.sqrt(hr_state[1, :, :, z_level]**2 + hr_state[2, :, :, z_level]**2).T, origin = "lower", extent = [0, 1, 0, 1])
axes[1,2].imshow(hr_state[4, :, :, z_level].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())

axes[2,0].imshow(sr_state[0, :, :, z_level_sr].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())
axes[2,1].imshow(np.sqrt(sr_state[1, :, :, z_level_sr]**2 + sr_state[2, :, :, z_level_sr]**2 + sr_state[3, :, :, z_level_sr]**2 ).T, origin = "lower", extent = [0, 1, 0, 1])
axes[2,2].imshow(sr_state[4, :, :, z_level_sr].T, origin = "lower", extent = [0, 1, 0, 1], norm = LogNorm())

axes[0,0].set_title("Density")
axes[0,1].set_title("Velocity")
axes[0,2].set_title("Pressure")